In [ ]:
import pandas as pd
import os
import requests
import base64

# === CONFIGURATION ===
csv_path = r"C:\GitHub\Android-Mobile-Apps\8.1-Project_GitHub_URLs.csv"
output_dir = r"C:\GitHub\AndroidProjects"
os.makedirs(output_dir, exist_ok=True)

# Subfolders
yml_output_dir = os.path.join(output_dir, "Config Files")
commits_dir = os.path.join(output_dir, "Commits")
build_info_dir = os.path.join(output_dir, "BuildInfo")
meta_output = os.path.join(output_dir, "Repo_Metadata.csv")

for d in [yml_output_dir, commits_dir, build_info_dir]:
    if os.path.exists(d):
        for file in os.listdir(d):
            os.remove(os.path.join(d, file))
    else:
        os.makedirs(d)

# === LOAD AND CLEAN CSV ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]

def extract_repo_info(url):
    parts = url.replace("https://github.com/", "").split("/")
    return parts[0], parts[1] if len(parts) > 1 else None

def github_api(url):
    headers = {'Accept': 'application/vnd.github.v3+json'}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    return None

def save_text(content_b64, path):
    try:
        decoded = base64.b64decode(content_b64).decode('utf-8')
        with open(path, 'w', encoding='utf-8') as f:
            f.write(decoded)
        return decoded
    except Exception:
        return ""

metadata = []

for url in df['github_url']:
    user, repo = extract_repo_info(url)
    print(f"\n📦 Processing {user}/{repo}...")

    # === Metadata
    repo_info = github_api(f"https://api.github.com/repos/{user}/{repo}")
    commits_info = github_api(f"https://api.github.com/repos/{user}/{repo}/commits")
    contributors_info = github_api(f"https://api.github.com/repos/{user}/{repo}/contributors")

    metadata.append({
        'repo': f"{user}/{repo}",
        'stars': repo_info.get('stargazers_count') if repo_info else None,
        'forks': repo_info.get('forks_count') if repo_info else None,
        'size_kb': repo_info.get('size') if repo_info else None,
        'commit_count': len(commits_info) if commits_info else None,
        'last_commit_date': commits_info[0]['commit']['author']['date'] if commits_info else None,
        'contributors': len(contributors_info) if contributors_info else None
    })

    # === Search and download .yml/.yaml in common CI folders
    ci_folders = ['.github/workflows', '.circleci', '.travis', '.gitlab']
    for folder in ci_folders:
        files = github_api(f"https://api.github.com/repos/{user}/{repo}/contents/{folder}")
        if not isinstance(files, list):
            continue
        for file in files:
            if file['name'].endswith(('.yml', '.yaml')):
                raw = github_api(file['url'])
                if raw and raw.get('content'):
                    save_text(raw['content'], os.path.join(yml_output_dir, f"{repo}.{file['name']}"))

    # === Search build.gradle(.kts) in root
    root_files = github_api(f"https://api.github.com/repos/{user}/{repo}/contents/")
    if isinstance(root_files, list):
        for file in root_files:
            if file['name'].startswith("build.gradle"):
                raw = github_api(file['url'])
                if raw and raw.get('content'):
                    decoded = save_text(raw['content'], os.path.join(build_info_dir, f"{repo}.{file['name']}"))
                    if 'test' in decoded.lower():
                        with open(os.path.join(build_info_dir, f"{repo}_test_lines.txt"), 'w', encoding='utf-8') as out:
                            for line in decoded.splitlines():
                                if 'test' in line.lower():
                                    out.write(line + "\n")

# === Export metadata
pd.DataFrame(metadata).to_csv(meta_output, index=False)
print("\n✅ All tasks completed without cloning.")
